<a href="https://colab.research.google.com/github/Fardous-bp/CNS-doped-Al-interconnect-alloy/blob/main/CNS_Al_18_75_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matcalc

!pip install matgl
!pip install seekpath

!pip install crystal-toolkit

In [ ]:
import os
from pymatgen.core import Structure
my_cif_file = "CuNi2Sn.cif"

if os.path.exists(my_cif_file):
    my_structure = Structure.from_file(my_cif_file)
    print(f"SUCCESS: {my_cif_file} loaded successfully.")
    print(my_structure)
else:
    print(f"ERROR: File '{my_cif_file}' not found. Please check the name in your Colab files tab.")

In [ ]:
# The value 0.2 adds random noise (in Angstroms) to the atomic sites
cuni2sn_perturbed = my_structure.copy()
cuni2sn_perturbed.perturb(0.2)

# 2. Expand the lattice volume
# Multiplying by 1.2 increases the total cell volume by 20%
cuni2sn_perturbed.scale_lattice(my_structure.volume * 1.2)

# 3. View the results
print("Perturbed CuNi2Sn Structure:")
print(cuni2sn_perturbed)
cuni2sn_perturbed

In [ ]:
import matcalc
from matcalc.utils import UNIVERSAL_CALCULATORS

import pprint
pprint.pprint(list(UNIVERSAL_CALCULATORS))  # calculators that come with bundled with matgl

In [ ]:
from matcalc.utils import MODEL_ALIASES
pprint.pprint(MODEL_ALIASES)  # list all "aliased" models

In [ ]:
calculator_pbe = matcalc.load_fp("pbe")
# calculator_pbe = matcalc.load_fp("TensorNet-MatPES-PBE-v2025.1-PES")

In [ ]:
# Initialize the Relaxer exactly like the notebook
relax_calc = matcalc.RelaxCalc(
    calculator_pbe,
    optimizer="FIRE",
    relax_atoms=True,
    relax_cell=True,
)

# This should now complete in 1-3 minutes
print("Starting structural optimization...")
data = relax_calc.calc(cuni2sn_perturbed)

# Output results
print(f"Optimization Successful!")
print(f"Final Energy: {data['energy']:.4f} eV")
print(f"Final Optimized Volume: {data['final_structure'].volume:.2f} A^3")

In [ ]:
pprint.pprint(data)

In [ ]:
final_structure_pbe = data["final_structure"]
print(final_structure_pbe)
final_structure_pbe

In [ ]:
from matcalc import ElasticityCalc

# Use the 'final_structure_pbe' generated
multiplier_GPa = 160.2176621
elastic_calc = ElasticityCalc(calculator_pbe, relax_structure=False)
elastic_results = elastic_calc.calc(final_structure_pbe)

print(f"Bulk Modulus: {elastic_results['bulk_modulus_vrh'] * multiplier_GPa:.2f} GPa")
print(f"Shear Modulus: {elastic_results['shear_modulus_vrh'] * multiplier_GPa:.2f} GPa")

In [ ]:
from matcalc import PhononCalc
import matplotlib.pyplot as plt

# 1. Initialize for the pure structure
phonon_calc = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for Pure CuNi2Sn...")
pure_phonon_data = phonon_calc.calc(final_structure_pbe)

# 2. Access the Phonopy object
ph = pure_phonon_data["phonon"]

# 3. Trigger the DOS calculation (Fixes the AttributeError)
# We use a mesh of 20x20x20
ph.run_mesh([20, 20, 20])
ph.run_total_dos()

# 4. Extract frequencies and densities
# get_total_dos_dict() returns a dictionary with 'frequency-points' and 'total-dos'
dos_dict = ph.get_total_dos_dict()
freqs = dos_dict['frequency_points']
densities = dos_dict['total_dos']

# 5. Visualization for Paper
plt.figure(figsize=(8, 5))
plt.plot(freqs, densities, color='blue', label='Pure CuNi2Sn')
plt.fill_between(freqs, densities, color='blue', alpha=0.2)
plt.ylim(0, 50)
plt.title("Phonon Density of States: Pure Baseline", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 1. Start with a fresh copy of your pure optimized CuNi2Sn structure
sac_al_18_75 = final_structure_pbe.copy()

# 2. Replace THREE sites with Al to reach 18.75% concentration
# We use indices 12 (Sn), 0 (Cu), and 4 (Ni) to keep the Al atoms separated
sac_al_18_75.replace(12, "Al")
sac_al_18_75.replace(0, "Al")
sac_al_18_75.replace(4, "Al")

print(f"Created 18.75% Al-doped structure: {sac_al_18_75.formula}")

# 3. High-Precision Relaxation
# We use the global relax_calc with fmax=0.001 for publication consistency
print("Starting high-precision optimization for 18.75% Al...")
relax_results_18_75 = relax_calc.calc(sac_al_18_75)
opt_struct_18_75 = relax_results_18_75["final_structure"]

print(f"18.75% Optimized Energy: {relax_results_18_75['energy']:.4f} eV")

In [ ]:
# 4. Calculate Elastic Moduli for the new concentration
elastic_results_18_75 = elastic_calc.calc(opt_struct_18_75)
multiplier_GPa = 160.2176621

bulk_18_75 = elastic_results_18_75['bulk_modulus_vrh'] * multiplier_GPa
shear_18_75 = elastic_results_18_75['shear_modulus_vrh'] * multiplier_GPa

print(f"--- 18.75% Al Doping Results ---")
print(f"Bulk Modulus: {bulk_18_75:.2f} GPa")
print(f"Shear Modulus: {shear_18_75:.2f} GPa")

# Monitor Pugh's Ratio for the brittleness argument
print(f"Pugh's Ratio (G/B): {shear_18_75 / bulk_18_75:.3f}")

In [ ]:
# 5. Initialize Phonon Calculator (2x2x2 supercell)
phonon_calc_18_75 = PhononCalc(
    calculator_pbe,
    supercell_matrix=((2, 0, 0), (0, 2, 0), (0, 0, 2)),
    relax_structure=False
)

print("Calculating Phonon Data for 18.75% Al-Doped CuNi2Sn...")
phonon_data_18_75 = phonon_calc_18_75.calc(opt_struct_18_75)
ph_18_75 = phonon_data_18_75["phonon"]

# Run DOS with high-mesh resolution
ph_18_75.run_mesh([20, 20, 20])
ph_18_75.run_total_dos()
dos_18_75 = ph_18_75.get_total_dos_dict()

# Visualization
plt.figure(figsize=(8, 5))
plt.plot(dos_18_75['frequency_points'], dos_18_75['total_dos'], color='purple', label='18.75% Al-Doped')
plt.fill_between(dos_18_75['frequency_points'], dos_18_75['total_dos'], color='purple', alpha=0.2)
plt.ylim(0, 50)
plt.title("Phonon DOS: 18.75% Al-Doped Baseline", fontsize=14)
plt.xlabel("Frequency (THz)", fontsize=12)
plt.ylabel("Density of States", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
import numpy as np

# Extract Thermodynamic Stability for 18.75% Al at Harsh Environment (473K)
ph_18_75.run_thermal_properties(t_step=5, t_max=600, t_min=0)
tp_18_75 = ph_18_75.get_thermal_properties_dict()

# Use np to find the index closest to your target temperature (473K)
idx_473 = (np.abs(tp_18_75['temperatures'] - 473)).argmin()

print(f"--- 18.75% Al THERMODYNAMICS ---")
print(f"Free Energy at 473K: {tp_18_75['free_energy'][idx_473]:.4f} kJ/mol")
print(f"Entropy at 473K: {tp_18_75['entropy'][idx_473]:.4f} J/K/mol")